# Phase 2: Full-Weight Supervised Fine-Tuning (SFT) Loop

This notebook contains the complete, enterprise-grade model tuning sequence to bake the past-tense narration, neutral academic voice, and Harvard/Ivey publishing standards directly into the model's weights.

**Hardware Requirement:** GPU (T4, L4, or A100/V100 in Google Colab).

### Step 1: Install Dependencies
We install the Hugging Face ecosystem libraries: `transformers`, `peft` (for handling model structures), `trl` (SFTTrainer), `bitsandbytes` (for quantization), and `datasets` (data parsing).

In [ ]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate sentencepiece

### Step 2: Import Libraries & Configure Setup

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

### Step 3: Set Model Targets and Quantization
We set our base target to Llama-3.1-8B-Instruct. We apply a 4-bit BitsAndBytes quantization config to fit the model in consumer/standard GPU memory (e.g. 16GB VRAM).

In [ ]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

### Step 4: Load the Custom Conversational Dataset
We load our compiled instruction dataset, mapping reference case styles.

In [ ]:
# Mount google drive or upload sft_training_dataset.json
dataset_file = "data/sft_training_dataset.json"

if not os.path.exists(dataset_file):
    print("Warning: File not found locally. Please upload 'sft_training_dataset.json' to the workspace.")
else:
    dataset = load_dataset("json", data_files=dataset_file, split="train")
    print(f"Loaded dataset with {len(dataset)} entries.")
    print(dataset[0])

### Step 5: Configure SFT Settings
To achieve a native past-tense voice, we configure a LoRA/Qlora script mapping all target modules, or prepare for full parameters if running on high-capacity Multi-A100 instances.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)

training_args = TrainingArguments(
    output_dir="./model_outputs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    report_to="none"
)

def format_prompts(batch):
    formatted_texts = []
    for messages in batch["messages"]:
        # Format messages into target chat template
        prompt = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted_texts.append(prompt)
    return {"text": formatted_texts}

dataset = dataset.map(format_prompts, batched=True)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=4096,
    tokenizer=tokenizer,
    args=training_args,
    packing=False
)

### Step 6: Execute Training Loop

In [ ]:
print("Starting Supervised Fine-Tuning (SFT)... ")
trainer.train()
print("Training Complete!")

### Step 7: Save Weights and Prepare for App Integration
We save the fine-tuned weights. The FastAPI app is configured to load these weights directly from the directory `./model/final_model` on startup.

In [ ]:
output_dir = "model/final_model"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Tuned weight binaries saved to directory: {output_dir}")